# furiosa-mlir playground

Triton written here compiles for, and runs on, the Furiosa NPU (npu3): `kernel[grid](...)` on CPU tensors
goes through the furiosa backend (raise to linalg, structure, tile, fopt, furiosa-opt crate) and the
results come back to the host. Nothing falls back to a CPU or GPU. Guide: `docs/usage.md`.

## 1. A kernel on the NPU

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    offs = tl.program_id(0) * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    tl.store(out_ptr + offs, tl.load(x_ptr + offs, mask=mask) + tl.load(y_ptr + offs, mask=mask), mask=mask)


n = 3000
x, y, out = torch.randn(n), torch.randn(n), torch.zeros(n)
add_kernel[(triton.cdiv(n, 1024),)](x, y, out, n, BLOCK=1024)  # first launch builds the crate (~1 min)
print("max abs error:", (out - (x + y)).abs().max().item())

## 2. Every stage of that launch

Each specialization keeps its stages under `~/.cache/furiosa-npu/<kernel>-<hash>/`.

In [ ]:
from furiosa_compiler.locks import CACHE

work = max(CACHE.glob("add_kernel-*"), key=lambda p: p.stat().st_mtime)
stages = sorted((work / "out" / "kernels").glob("*/*.mlir"))
for path in stages:
    print(path.relative_to(work))
print(next(p for p in stages if p.name == "linalg.mlir").read_text()[:3000])

## 3. A model: compile, run, trace

A model file defines `graphs()`, its launches in order (`models/*.py`). `furiosa trace --npu` runs every
launch on the CPU reference and on npu3 and prints each written buffer's error; `--dump` keeps them all.

In [ ]:
!furiosa compile models/lenet5.py -o out/lenet5

In [ ]:
!furiosa run out/lenet5

In [ ]:
!furiosa trace models/lenet5.py -o out/lenet5-trace --npu --dump out/lenet5-trace/dump

In [ ]:
from furiosa_compiler import tensors

dump = tensors.load("out/lenet5-trace/dump/000_conv2d_tanh_kernel_0.safetensors")
npu = tensors.load("out/lenet5-trace/dump/000_conv2d_tanh_kernel_0.npu.safetensors")
for name in dump:
    print(name, dump[name].shape, "max |npu - reference|:", float(abs(npu[name] - dump[name]).max()))